In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

corpus_dir = '/content/drive/MyDrive/Mémoire M1/Corpus '  # chemin corrigé

annotation_dirs = {
    'spaCy_sm': '/content/drive/MyDrive/Mémoire M1/Modèle : SpaCy/Sortie2sm',
    'spaCy_lg': '/content/drive/MyDrive/Mémoire M1/Modèle : SpaCy/Sortie2lg'
}


def read_tokens(file_path):
    tokens = []
    with open(file_path, 'r', encoding='latin-1') as f:
        for line in f:
            line = line.strip()
            if line:
                tokens.append(line)
    return tokens

def read_annotations(file_path):
    annotations = []
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                annotations.append(parts[1])
            else:
                annotations.append('O')
    return annotations

def convert_iob2_to_iob1(tags):
    new_tags = []
    prev_tag = 'O'
    prev_type = ''
    for tag in tags:
        if tag == 'O':
            new_tags.append('O')
            prev_tag = 'O'
            prev_type = ''
            continue

        tag_prefix, tag_type = tag.split('-', 1)

        if tag_prefix == 'B':
            if prev_tag != 'O' and prev_type == tag_type:
                new_tags.append('B-' + tag_type)
            else:
                new_tags.append('I-' + tag_type)
            prev_tag = tag_prefix
            prev_type = tag_type
        else:
            new_tags.append(tag)
            prev_tag = tag_prefix
            prev_type = tag_type

    return new_tags

all_tokens = []
all_tags = {model: [] for model in annotation_dirs}

corpus_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith('.txt')])

for filename in corpus_files:
    corpus_path = os.path.join(corpus_dir, filename)

    tokens = read_tokens(corpus_path)
    all_tokens.extend(tokens)

    for model_name, ann_dir in annotation_dirs.items():
        ann_path = os.path.join(ann_dir, filename)

        if os.path.exists(ann_path):
            tags = read_annotations(ann_path)
            if len(tags) != len(tokens):
                print(f"⚠️ Mismatch: {filename} pour {model_name} ({len(tags)} vs {len(tokens)})")
            all_tags[model_name].extend(tags)
        else:
            print(f"⚠️ Annotation manquante pour {filename} dans {model_name}")
            all_tags[model_name].extend(['O'] * len(tokens))

df = pd.DataFrame({'Token': all_tokens})

for model_name in annotation_dirs.keys():
    tags_iob2 = all_tags[model_name]
    df[f"{model_name}_IOB2"] = tags_iob2

    tags_iob1 = convert_iob2_to_iob1(tags_iob2)
    df[f"{model_name}_IOB1"] = tags_iob1

output_path = '/content/drive/MyDrive/comparaison_iob_spacy.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ Tableau comparatif exporté avec succès !\nFichier : {output_path}")
